In [118]:
import requests, xml.etree.ElementTree as ET, json

sitemap_urls = [
    'https://hoatuoimymy.com/product-sitemap0.xml',
    'https://hoatuoimymy.com/product-sitemap1.xml',
    'https://hoatuoimymy.com/product-sitemap2.xml',
    'https://hoatuoimymy.com/product-sitemap3.xml'
]

all_urls = []
for url in sitemap_urls:
    try:
        r = requests.get(url)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        all_urls += [loc.text for loc in root.iter('{http://www.sitemaps.org/schemas/sitemap/0.9}loc')]
    except Exception as e:
        print(f"Lỗi với {url}: {e}")

with open('all_urls.json', 'w') as f:
    json.dump(all_urls, f, indent=4)

print(f"Extracted  {len(all_urls)} URLs and saved to all_urls.json")


Extracted  741 URLs and saved to all_urls.json


Danh sách các trang web

In [119]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
from tqdm.notebook import tqdm

with open('all_urls.json', 'r', encoding='utf-8') as f:
    urls = json.load(f)

data = []

for url in tqdm(urls):
    try:
        resp = requests.get(url, timeout=10)
        soup = BeautifulSoup(resp.content, 'html.parser')
        
        # Lấy title, price như cũ
        title = soup.select_one('h1.product_title')
        price = soup.select_one('.price span.amount')
        
        # Lấy mô tả sản phẩm (description) - lấy cả nhiều nguồn
        desc = None
        # 1. WooCommerce short description
        desc_tag = soup.select_one('.woocommerce-product-details__short-description')
        if desc_tag:
            desc = desc_tag.get_text(separator=' ', strip=True)
        # 2. Thử lấy từ .product-short-description
        if not desc:
            desc_tag2 = soup.select_one('.product-short-description')
            if desc_tag2:
                desc = desc_tag2.get_text(separator=' ', strip=True)
        # 3. Thử lấy từ meta description
        if not desc:
            meta_desc = soup.find('meta', attrs={'name': 'description'})
            if meta_desc and meta_desc.has_attr('content'):
                desc = meta_desc['content']
        
        # Lấy thông tin khuyến mãi (nếu có)
        khuyen_mai = None
        km_tag = soup.select_one('div.khuyen-mai')
        if km_tag:
            # Lấy text của các <li> trong khuyến mãi
            km_items = km_tag.find_all('li')
            if km_items:
                khuyen_mai = "; ".join([li.get_text(separator=' ', strip=True) for li in km_items])
            else:
                khuyen_mai = km_tag.get_text(separator=' ', strip=True)
        
        # Không gộp khuyến mãi vào description nữa, để riêng
        full_desc = desc if desc else None
        
        # Lấy ảnh: thử nhiều selector
        image = None
        # 1. Ảnh trong gallery
        img_tag = soup.select_one('.woocommerce-product-gallery__image img')
        if img_tag and img_tag.has_attr('src'):
            image = img_tag['src']
        # 2. Ảnh đại diện sản phẩm
        if not image:
            img_tag = soup.select_one('img.wp-post-image')
            if img_tag and img_tag.has_attr('src'):
                image = img_tag['src']
        # 3. Ảnh trong meta property
        if not image:
            meta_img = soup.find('meta', property='og:image')
            if meta_img and meta_img.has_attr('content'):
                image = meta_img['content']
        
        data.append({
            'url': url,
            'title': title.get_text(strip=True) if title else None,
            'price': price.get_text(strip=True) if price else None,
            'description': full_desc,
            'khuyen_mai': khuyen_mai,
            'image': image
        })
    except Exception as e:
        print(f"Lỗi với {url}: {e}")

df = pd.DataFrame(data)
df.to_csv('products.csv', index=False, encoding='utf-8-sig')
print("Đã lưu dữ liệu ra products.csv")

  0%|          | 0/741 [00:00<?, ?it/s]

Đã lưu dữ liệu ra products.csv


In [120]:
# read_CSV
df = pd.read_csv('products.csv')
# Display the first few rows of the DataFrame   
df

,url,title,price,description,khuyen_mai,image
0,https://hoatuoimymy.com/cua-hang/,NaN,2.400.000₫,Sản phẩm Archive - Shop Hoa Tươi My My,NaN,/wp-content/uploads/2023/12/banner-1_0.jpg
1,https://hoatuoimymy.com/hoa-chia-buon-m20/,Hoa Chia Buồn M20,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
2,https://hoatuoimymy.com/hoa-dam-tang-m503/,Hoa Đám Tang M503,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
3,https://hoatuoimymy.com/hoa-khai-truong-m433/,Hoa Khai Trương M433,1.100.000₫,Hoa Khai Trương M433 là món quà tuyệt vời dành...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
4,https://hoatuoimymy.com/gio-hoa-m472/,Giỏ Hoa M472,970.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
...,...,...,...,...,...,...
736,https://hoatuoimymy.com/bo-hoa-huong-duong-m114/,Bó Hoa Hướng Dương M114,650.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
737,https://hoatuoimymy.com/bo-hoa-baby-m115/,Bó Hoa Baby M115,700.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
738,https://hoatuoimymy.com/bo-hoa-hong-m116/,Bó Hoa Hồng M116,700.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
739,https://hoatuoimymy.com/bo-hoa-m117/,Bó Hoa M117,650.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...


In [121]:
df.head()

,url,title,price,description,khuyen_mai,image
0,https://hoatuoimymy.com/cua-hang/,NaN,2.400.000₫,Sản phẩm Archive - Shop Hoa Tươi My My,NaN,/wp-content/uploads/2023/12/banner-1_0.jpg
1,https://hoatuoimymy.com/hoa-chia-buon-m20/,Hoa Chia Buồn M20,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
2,https://hoatuoimymy.com/hoa-dam-tang-m503/,Hoa Đám Tang M503,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
3,https://hoatuoimymy.com/hoa-khai-truong-m433/,Hoa Khai Trương M433,1.100.000₫,Hoa Khai Trương M433 là món quà tuyệt vời dành...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
4,https://hoatuoimymy.com/gio-hoa-m472/,Giỏ Hoa M472,970.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...


In [122]:
from dotenv import load_dotenv
load_dotenv()
import os
import sys
import google.generativeai as genai
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance



In [123]:
url = os.getenv("QDRANT_ENDPOINT")
key = os.getenv("QDRANT_API_KEY")

client = QdrantClient(
        url=url,
        api_key=key
    )

2025-07-18 05:06:08 [httpx] INFO: HTTP Request: GET https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"


In [124]:
client.get_collections()

2025-07-18 05:06:09 [httpx] INFO: HTTP Request: GET https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections "HTTP/1.1 200 OK"


CollectionsResponse(collections=[])

In [125]:
# create collection 
collection_name = "RAG_Huy"
if not client.collection_exists(collection_name):
    client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=768, distance=Distance.COSINE, on_disk=True),
            shard_number=2,
            timeout=180
        )
    print(f"Collection '{collection_name}' created.")
else:
    print(f"Collection '{collection_name}' already exists.")

2025-07-18 05:06:09 [httpx] INFO: HTTP Request: GET https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/RAG_Huy/exists "HTTP/1.1 200 OK"
2025-07-18 05:06:10 [httpx] INFO: HTTP Request: PUT https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/RAG_Huy?timeout=180 "HTTP/1.1 200 OK"


Collection 'RAG_Huy' created.


In [126]:
model = genai.GenerativeModel('gemini-2.0-flash')


In [127]:

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("Alibaba-NLP/gte-multilingual-base", 
                                      trust_remote_code=True)

2025-07-18 05:06:10 [sentence_transformers.SentenceTransformer] INFO: Use pytorch device_name: cpu
2025-07-18 05:06:10 [sentence_transformers.SentenceTransformer] INFO: Load pretrained SentenceTransformer: Alibaba-NLP/gte-multilingual-base
Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [128]:
def get_vector(text):
    """
    Generate an embedding for the given text using the preloaded model.

    Args:
        text (str): The input text to encode.

    Returns:
        list: The embedding as a list of floats, or an empty list if input is empty.
    """
    if not text.strip():
        print("Attempted to get embedding for empty text.")
        return []

    embedding = embedding_model.encode(text)
    return embedding.tolist()

In [129]:
len(get_vector("Hello world")) # Test embedding function

768

In [130]:
df.head()

,url,title,price,description,khuyen_mai,image
0,https://hoatuoimymy.com/cua-hang/,NaN,2.400.000₫,Sản phẩm Archive - Shop Hoa Tươi My My,NaN,/wp-content/uploads/2023/12/banner-1_0.jpg
1,https://hoatuoimymy.com/hoa-chia-buon-m20/,Hoa Chia Buồn M20,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
2,https://hoatuoimymy.com/hoa-dam-tang-m503/,Hoa Đám Tang M503,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
3,https://hoatuoimymy.com/hoa-khai-truong-m433/,Hoa Khai Trương M433,1.100.000₫,Hoa Khai Trương M433 là món quà tuyệt vời dành...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...
4,https://hoatuoimymy.com/gio-hoa-m472/,Giỏ Hoa M472,970.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...


In [131]:
df['content'] = df['title'] +  ' giá ' + df['price'] + ' mô tả sản phẩm: ' + df['description'] + ' khuyêt mãi: ' + df['khuyen_mai'] + ' xem ảnh tại ' + df['image'].fillna('') 
df.head()

,url,title,price,description,khuyen_mai,image,content
0,https://hoatuoimymy.com/cua-hang/,NaN,2.400.000₫,Sản phẩm Archive - Shop Hoa Tươi My My,NaN,/wp-content/uploads/2023/12/banner-1_0.jpg,NaN
1,https://hoatuoimymy.com/hoa-chia-buon-m20/,Hoa Chia Buồn M20,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...,Hoa Chia Buồn M20 giá 1.300.000₫ mô tả sản phẩ...
2,https://hoatuoimymy.com/hoa-dam-tang-m503/,Hoa Đám Tang M503,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...,Hoa Đám Tang M503 giá 1.300.000₫ mô tả sản phẩ...
3,https://hoatuoimymy.com/hoa-khai-truong-m433/,Hoa Khai Trương M433,1.100.000₫,Hoa Khai Trương M433 là món quà tuyệt vời dành...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...,Hoa Khai Trương M433 giá 1.100.000₫ mô tả sản ...
4,https://hoatuoimymy.com/gio-hoa-m472/,Giỏ Hoa M472,970.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...,Giỏ Hoa M472 giá 970.000₫ mô tả sản phẩm: Flow...


In [132]:
df['vector'] = df['content'].astype(str).apply(lambda x: get_vector(x) if isinstance(x, str) else [])
df.head()

,url,title,price,description,khuyen_mai,image,content,vector
0,https://hoatuoimymy.com/cua-hang/,NaN,2.400.000₫,Sản phẩm Archive - Shop Hoa Tươi My My,NaN,/wp-content/uploads/2023/12/banner-1_0.jpg,NaN,"[-0.06145051121711731, 0.043867360800504684, 0..."
1,https://hoatuoimymy.com/hoa-chia-buon-m20/,Hoa Chia Buồn M20,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...,Hoa Chia Buồn M20 giá 1.300.000₫ mô tả sản phẩ...,"[-0.04165337607264519, 0.005678410641849041, -..."
2,https://hoatuoimymy.com/hoa-dam-tang-m503/,Hoa Đám Tang M503,1.300.000₫,Freeship nội thành. Tư vấn nhiệt tình. Nhận th...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...,Hoa Đám Tang M503 giá 1.300.000₫ mô tả sản phẩ...,"[-0.030913934111595154, 0.04950922355055809, -..."
3,https://hoatuoimymy.com/hoa-khai-truong-m433/,Hoa Khai Trương M433,1.100.000₫,Hoa Khai Trương M433 là món quà tuyệt vời dành...,Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...,Hoa Khai Trương M433 giá 1.100.000₫ mô tả sản ...,"[-0.053939566016197205, 0.07622317969799042, -..."
4,https://hoatuoimymy.com/gio-hoa-m472/,Giỏ Hoa M472,970.000₫,"Flower Mẫu hoa sang trọng, tinh tế. Freeship n...",Miễn phí giao hàng tại khu vực nội thành với h...,https://hoatuoimymy.com/wp-content/uploads/202...,Giỏ Hoa M472 giá 970.000₫ mô tả sản phẩm: Flow...,"[-0.0647134780883789, 0.03957376629114151, -0...."


In [133]:
import qdrant_client
from qdrant_client.http import models as qdrant_models
import uuid

collection_name = "RAG_Huy"

# To avoid WriteTimeout, split the upsert into smaller batches
import math

BATCH_SIZE = 50  # You can adjust this value as needed

def url_to_uuid(url):
    """
    Convert a URL string to a UUID using uuid5 and a fixed namespace.
    This ensures the same URL always maps to the same UUID.
    """
    return str(uuid.uuid5(uuid.NAMESPACE_URL, str(url)))

points = []
for idx, row in df.iterrows():
    vector = row.get("vector")
    payload = {
        "url": row.get("url"),
        "title": row.get("title"),
        "price": row.get("price"),
        "description": row.get("description"),
        "khuyen_mai": row.get("khuyen_mai"),
        "image": row.get("image"),
        "id": int(row.get("id")) if not pd.isna(row.get("id")) else None
    }
    # Qdrant requires point id to be an unsigned integer or a UUID.
    # We'll use a UUID generated from the URL for uniqueness and compliance.
    url_val = row.get("url")
    point_id = url_to_uuid(url_val) if url_val else str(uuid.uuid4())
    points.append(
        qdrant_models.PointStruct(
            id=point_id,
            vector=vector,
            payload=payload
        )
    )

# Upsert in batches to avoid timeouts
total_points = len(points)
num_batches = math.ceil(total_points / BATCH_SIZE)

for i in range(num_batches):
    batch_points = points[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
    try:
        client.upsert(
            collection_name=collection_name,
            points=batch_points,
        )
        print(f"Upserted batch {i+1}/{num_batches} ({len(batch_points)} points) to Qdrant collection '{collection_name}'.")
    except Exception as e:
        print(f"Error upserting batch {i+1}: {e}")

print(f"Finished upserting {total_points} points to Qdrant collection '{collection_name}'.")


2025-07-18 05:10:08 [httpx] INFO: HTTP Request: PUT https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/RAG_Huy/points?wait=true "HTTP/1.1 200 OK"


Upserted batch 1/15 (50 points) to Qdrant collection 'RAG_Huy'.


2025-07-18 05:10:10 [httpx] INFO: HTTP Request: PUT https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/RAG_Huy/points?wait=true "HTTP/1.1 200 OK"


Upserted batch 2/15 (50 points) to Qdrant collection 'RAG_Huy'.


2025-07-18 05:10:11 [httpx] INFO: HTTP Request: PUT https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/RAG_Huy/points?wait=true "HTTP/1.1 200 OK"


...

Upserted batch 13/15 (50 points) to Qdrant collection 'RAG_Huy'.


2025-07-18 05:10:25 [httpx] INFO: HTTP Request: PUT https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/RAG_Huy/points?wait=true "HTTP/1.1 200 OK"


Upserted batch 14/15 (50 points) to Qdrant collection 'RAG_Huy'.


2025-07-18 05:10:26 [httpx] INFO: HTTP Request: PUT https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/RAG_Huy/points?wait=true "HTTP/1.1 200 OK"


Upserted batch 15/15 (41 points) to Qdrant collection 'RAG_Huy'.
Finished upserting 741 points to Qdrant collection 'RAG_Huy'.


In [137]:
#  query
query = "hoa tươi sinh nhật"
query_vector = get_vector(query)
search_result = client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=5,  # Limit the number of results
    with_payload=True  # Include payload in the results
)

for record in search_result:
    print(f" Payload: {record.payload}, score: {record.score}")
    

2025-07-18 05:11:10 [py.warnings] WARNING: C:\Users\VUHUY\AppData\Local\Temp\ipykernel_20840\1148571933.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = client.search(

2025-07-18 05:11:11 [httpx] INFO: HTTP Request: POST https://a6f0b5da-8041-45e8-b56a-d2a779c1520f.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/RAG_Huy/points/search "HTTP/1.1 200 OK"


 Payload: {'url': 'https://hoatuoimymy.com/gio-hoa-sinh-nhat-vip-m155/', 'title': 'Giỏ Hoa Sinh Nhật Vip M155', 'price': '1.950.000₫', 'description': 'Flower Mẫu hoa sang trọng, tinh tế. Freeship nội thành. Tư vấn nhiệt tình. Nhận thiết kế lẵng hoa theo ngân sách của khách hàng. Dịch vụ: Giao Gấp Trong Vòng 40 phút', 'khuyen_mai': 'Miễn phí giao hàng tại khu vực nội thành với hóa đơn từ 400K trở lên; Tặng Banner Hoặc Thiệp (Trị Giá 20.000đ – 50.000đ); Có Giao Nhanh Trong Vòng 1 - 2 Giờ; Hoàn Lại Tiền Nếu Khách Nhận Hoa Không Như Cam Kết; Hoa Tươi Trên 3 Ngày; Giảm Tiếp 5% Cho Đơn Hàng Bạn Tạo ONLINE Lần Thứ 2 trong tuần', 'image': 'https://hoatuoimymy.com/wp-content/uploads/2024/08/z5734688705869_ae52345335d0de344f1d1207fbcd18de-1-500x565.jpg', 'id': None}, score: 0.70793176
 Payload: {'url': 'https://hoatuoimymy.com/lang-hoa-sinh-nhat-m65/', 'title': 'Lẵng Hoa Sinh Nhật M65', 'price': '750.000₫', 'description': 'Flower Mẫu hoa sang trọng, tinh tế. Freeship nội thành. Tư vấn nhiệt tình